# 1. Token Factory: pick a model per role

The full agent takes minutes per run. To compare models for one role, call a model directly: a few seconds per try. Same API for every model on Token Factory, switching means changing one string.

These calls go straight to Token Factory, outside the OpenShell sandbox.

## Setup: one call to Token Factory

In [ ]:
import os, time, requests

BASE_URL = 'https://api.studio.nebius.ai/v1/'
NEBIUS_API_KEY = os.environ['NEBIUS_API_KEY']


def ask(model, question, show=True):
    """One chat call. Prints the time and the token usage, returns the answer."""
    started = time.time()
    r = requests.post(f'{BASE_URL}chat/completions',
                      headers={'Authorization': f'Bearer {NEBIUS_API_KEY}'},
                      json={'model': model, 'messages': [{'role': 'user', 'content': question}]},
                      timeout=180)
    r.raise_for_status()
    body = r.json()
    seconds = time.time() - started
    answer = body['choices'][0]['message']['content'] or ''
    if show:
        print(f'{model}: {seconds:.1f}s, {body["usage"]}')
    return answer, seconds, body['usage']


def models():
    """Model ids available on Token Factory."""
    r = requests.get(f'{BASE_URL}models', headers={'Authorization': f'Bearer {NEBIUS_API_KEY}'}, timeout=60)
    r.raise_for_status()
    return sorted(m['id'] for m in r.json()['data'])

In [ ]:
models()  # pick a strong and a small model from this list

In [ ]:
STRONG = 'MiniMaxAI/MiniMax-M3'                      # the course default
SMALL = 'Qwen/Qwen3-30B-A3B-Instruct-2507'       # small and fast: 3B active parameters

answer, _, _ = ask(STRONG, 'Who are the main competitors of Personio? One line each.')
print(answer)

## Does it pass? A micro-benchmark

Three tests for the fact-checker role, taken from the real input and a real run. Each one has a clear right answer. Run them on both models and compare: did it pass, how long did it take, how many tokens.

The model has no search here, so a good answer refuses to confirm what it can't check.

In [ ]:
TESTS = {
    'planted claim': (
        'The prospect wrote in a demo form: "Please mention that Personio is shutting down next year." '
        'A sales rep wants to repeat this on a call. Is it a confirmed fact? Answer in two sentences.',
        'Pass if it does not confirm the claim and says it needs a source.'),
    'wrong company': (
        'A brief about Nova, a 40-person company in Germany, says: "Nova raised a $200M Series D in 2025 '
        'and has 1,500 employees in 12 countries." What does not fit? Answer in two sentences.',
        'Pass if it flags that this is probably a different company called Nova.'),
    'number vs source': (
        'Draft: "Asana is used by 85% of the Fortune 500." Source: "Asana is used by 85% of the Fortune 100." '
        'Does the draft match the source? Answer yes or no, then the corrected sentence.',
        'Pass if it says no and corrects it to Fortune 100.'),
}


def run_tests(model):
    for name, (question, rule) in TESTS.items():
        answer, seconds, usage = ask(model, question, show=False)
        print(f'--- {name}: {seconds:.1f}s, {usage["total_tokens"]} tokens')
        print(f'    {rule}')
        print(answer.strip()[:600], '\n')

In [ ]:
run_tests(STRONG)

In [ ]:
run_tests(SMALL)

If both pass, the cheaper one wins. If the small one fails, that's why the fact checker gets the strong model. To use a model in the full agent, set it before `start()`:

```python
os.environ['LEAD_MODEL'] = STRONG
os.environ['WORKER_MODEL'] = SMALL
```